# Progress Report #3: Evaluating Model Performance
**Project:** Cybersecurity Vulnerability Severity Prediction Using Graph Machine Learning  
**Team:** Chesda Saroth, Tabitha Ondieki, Abdullahi Oyeniyi  
**Dataset:** Cleaned UCKG CVE sample (`../dataset/cve_clean.parquet`)  

We train three models, compare train/test splits with 5-fold cross-validation, use bootstrap, and compare adjusted R². **Run All** from the repository's `code/` folder. The notebook uses the same CVE dataset as our previous work; no data is downloaded automatically.

## 0. Load and prepare the data
We import the libraries and load the cleaned vulnerability records. We then make six numeric attributes from each CVE description. The `description_length` feature is a character count; the `word_count` feature counts words. The other four fields count letters, digits, spaces and other characters.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, cross_validate, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

SEED = 42
df = pd.read_parquet("../dataset/cve_clean.parquet")
df["description"] = df["description"].fillna("").astype(str)
df = df[df["description"].str.strip() != ""].copy()
df["description_length"] = df["description"].str.len()
df["word_count"] = df["description"].str.split().str.len()
df["letter_count"] = df["description"].str.count("[A-Za-z]")
df["digit_count"] = df["description"].str.count(r"\d")
df["whitespace_count"] = df["description"].str.count(r"\s")
df["other_count"] = (df["description_length"] - df["letter_count"]
                     - df["digit_count"] - df["whitespace_count"])
df["severity"] = df["severity"].astype(str).str.upper().str.strip()
df = df[df["severity"].isin(["LOW", "MEDIUM", "HIGH", "CRITICAL"])].reset_index(drop=True)
print("Records:", len(df))
print("Severity counts:", df["severity"].value_counts().to_dict())
print(df[["description_length", "word_count", "letter_count"]].head())

**Interpretation:** We are working with numeric features rather than raw text. The four character categories add up to total description length; this will explain why the multiple linear regression below has an unusually good fit.

## 1. Simple linear regression
**Attributes:** X = `description_length`, y = `word_count`. We use 80% of the records for training and 20% for testing. `fit()` trains the model, `predict()` estimates word counts, and MSE/RMSE/MAE measure prediction error. R² measures explained variation.

In [ ]:
X1 = df[["description_length"]]
y1 = df["word_count"]
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=SEED)
model1 = LinearRegression().fit(X1_train, y1_train)
pred1 = model1.predict(X1_test)

mse1 = mean_squared_error(y1_test, pred1)
print(f"R² = {r2_score(y1_test, pred1):.4f}")
print(f"MSE = {mse1:.4f}, RMSE = {np.sqrt(mse1):.4f}, MAE = {mean_absolute_error(y1_test, pred1):.4f}")

plt.figure(figsize=(8, 4))
plt.scatter(X1_test["description_length"], y1_test, alpha=0.5, label="Observed")
order = np.argsort(X1_test["description_length"].to_numpy())
plt.plot(X1_test["description_length"].to_numpy()[order], pred1[order], color="orange", label="Predicted")
plt.xlabel("Description length (characters)")
plt.ylabel("Word count")
plt.title("Simple regression: observed and predicted")
plt.legend()
plt.show()

**Interpretation:** Descriptions with more characters tend to have more words. A high test R² and low error values mean the line is a good estimate, although individual descriptions may differ in average word length.

## 2. Multiple linear regression
**Attributes:** Four inputs (`letter_count`, `digit_count`, `whitespace_count`, `other_count`); target = `description_length`. The training and error calculations are the same as Section 1. The 3D scatter displays only two of the four inputs because 3D graphs have three axes.

In [ ]:
features = ["letter_count", "digit_count", "whitespace_count", "other_count"]
X2 = df[features]
y2 = df["description_length"]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=SEED)
model2 = LinearRegression().fit(X2_train, y2_train)
pred2 = model2.predict(X2_test)
mse2 = mean_squared_error(y2_test, pred2)
print(f"R² = {r2_score(y2_test, pred2):.4f}")
print(f"MSE = {mse2:.6f}, RMSE = {np.sqrt(mse2):.6f}, MAE = {mean_absolute_error(y2_test, pred2):.6f}")

fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(X2_test["letter_count"], X2_test["whitespace_count"], y2_test, label="Observed", alpha=0.6)
ax.scatter(X2_test["letter_count"], X2_test["whitespace_count"], pred2, label="Predicted", marker="^", alpha=0.6)
ax.set(xlabel="Letters", ylabel="Spaces", zlabel="Total characters", title="Multiple regression")
ax.legend()
plt.show()

**Interpretation:** R² should be 1.0 with errors close to zero **because the four inputs add up exactly to the target**. This is a useful demonstration of regression, not evidence that the model predicts an independent cybersecurity outcome.

## 3. Logistic regression
Unlike the first two models, logistic regression predicts a **category** (LOW / MEDIUM / HIGH). We split each CVSS vector into its individual components and one-hot encode them in a pipeline before fitting `LogisticRegression`. We report accuracy, **absolute loss** (mean absolute error after ordering severity from LOW to CRITICAL), 0–1 error, and log loss (which also scores predicted probabilities).

In [ ]:
# Example: "AV:N/AC:L/Au:N" becomes {"AV": "N", "AC": "L", "Au": "N"}.
X3 = df["cvss_vector"].fillna("").apply(
    lambda s: dict(part.split(":", 1) for part in str(s).split("/") if ":" in part)
).apply(pd.Series).fillna("missing")
y3 = df["severity"]
X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=SEED, stratify=y3
)
model3 = make_pipeline(OneHotEncoder(handle_unknown="ignore"), LogisticRegression(max_iter=3000))
model3.fit(X3_train, y3_train)
pred3 = model3.predict(X3_test)
prob3 = model3.predict_proba(X3_test)

acc3 = accuracy_score(y3_test, pred3)
severity_values = {"LOW": 0, "MEDIUM": 1, "HIGH": 2, "CRITICAL": 3}
abs_loss = mean_absolute_error(y3_test.map(severity_values), [severity_values[s] for s in pred3])
print(f"Accuracy = {acc3:.4f}")
print(f"Absolute loss (ordinal MAE) = {abs_loss:.4f}")
print(f"0-1 error = {1 - acc3:.4f}")
print(f"Log loss = {log_loss(y3_test, prob3, labels=model3.classes_):.4f}")

levels = [s for s in ["LOW", "MEDIUM", "HIGH", "CRITICAL"] if s in set(y3)]
lookup = {s: i for i, s in enumerate(levels)}
plt.figure(figsize=(9, 3.5))
plt.scatter(range(len(y3_test)), y3_test.map(lookup), label="Observed", alpha=0.5)
plt.scatter(range(len(pred3)), [lookup[s] for s in pred3], label="Predicted", marker="x", alpha=0.7)
plt.yticks(range(len(levels)), levels)
plt.xlabel("Test record")
plt.ylabel("Severity")
plt.title("Logistic regression: observed and predicted")
plt.legend()
plt.show()

**Interpretation:** Markers at the same severity level represent correct classifications. Compare accuracy against the largest class's share to check that the model beats a majority-class guess. CVSS metrics are used to define CVSS severity, so a strong score here is **partly built into the problem**; this is not independent prediction of exploit likelihood.

## 4. Cross-validation vs. train/test split
A single 80/20 split gives one score. **Five-fold cross-validation** repeats training with five different validation portions. Here, `cross_validate()` gives the mean and standard deviation of the evaluation score. The logistic model uses stratified folds to preserve class proportions.

In [ ]:
folds = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv1 = cross_validate(LinearRegression(), X1, y1, cv=folds, scoring="r2")["test_score"]
cv2 = cross_validate(LinearRegression(), X2, y2, cv=folds, scoring="r2")["test_score"]
cv3 = cross_validate(model3, X3, y3, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                     scoring="accuracy")["test_score"]

compare = pd.DataFrame({
    "Model": ["Simple linear (R²)", "Multiple linear (R²)", "Logistic (accuracy)"],
    "Train/test": [r2_score(y1_test, pred1), r2_score(y2_test, pred2), acc3],
    "CV mean": [cv1.mean(), cv2.mean(), cv3.mean()],
    "CV SD": [cv1.std(), cv2.std(), cv3.std()]
})
print(compare.round(4).to_string(index=False))

**Interpretation:** Compare the `Train/test` column to `CV mean`. Similar values mean the result is not unusually dependent on one split. A smaller `CV SD` indicates more consistent performance. R² and accuracy apply to different tasks and should not be compared directly against each other.

## 5. Bootstrap the simple regression
We draw **200 random training samples with replacement**, retrain the simple linear model on each sample, and calculate MAE on the same held-out test set. The mean summarizes typical error and standard deviation (SD) measures variation between resamples.

In [ ]:
rng = np.random.default_rng(SEED)
errors = []
for _ in range(200):
    sample = rng.integers(0, len(X1_train), size=len(X1_train))
    boot_model = LinearRegression().fit(X1_train.iloc[sample], y1_train.iloc[sample])
    errors.append(mean_absolute_error(y1_test, boot_model.predict(X1_test)))

print(f"Bootstrap MAE mean = {np.mean(errors):.4f} words")
print(f"Bootstrap MAE SD = {np.std(errors, ddof=1):.4f} words")
plt.figure(figsize=(7, 3.5))
plt.hist(errors, bins=20, edgecolor="white")
plt.xlabel("Test MAE (words)")
plt.ylabel("Number of resamples")
plt.title("Bootstrap error distribution")
plt.show()

**Interpretation:** The mean bootstrap MAE should be close to the simple model's test MAE. The SD tells us how sensitive the fitted model is to choosing different training observations. This bootstrap varies the **training set**, while the test set stays fixed.

## 6. Compare models with adjusted R²
Adjusted R² rewards a good fit but penalizes extra predictors. To compare fairly, **both models must predict the same target**: `description_length`. We compare using letters alone versus using all four character categories, on the same training observations.

In [ ]:
one_feature = LinearRegression().fit(X2_train[["letter_count"]], y2_train)
r2_one = r2_score(y2_train, one_feature.predict(X2_train[["letter_count"]]))
r2_four = r2_score(y2_train, model2.predict(X2_train))
n = len(y2_train)
adj_one = 1 - (1 - r2_one) * (n - 1) / (n - 2)
adj_four = 1 - (1 - r2_four) * (n - 1) / (n - 5)

print(f"One predictor adjusted R² = {adj_one:.4f}")
print(f"Four predictors adjusted R² = {adj_four:.4f}")
print("Higher adjusted R²:", "four predictors" if adj_four > adj_one else "one predictor")

**Interpretation:** If the four-feature adjusted R² is larger, the improvement in fit outweighs the penalty for three extra predictors. In this case the gain is unsurprising: the four character counts define the target exactly.

## 7. Conclusion
We trained and evaluated **simple linear regression, multiple linear regression, and logistic regression** using the cleaned UCKG CVE dataset.

- The simple model measured the relationship between description length and word count using R², MSE, RMSE, MAE, and a prediction plot.
- The multiple model used four attributes and a 3D observed-versus-predicted plot. Its nearly perfect fit is a direct character-count identity rather than independent prediction.
- The logistic model predicted CVE severity from CVSS fields using accuracy, ordinal absolute loss, 0–1 error, log loss, and a prediction plot.
- Five-fold cross-validation checked the stability of all three models against the initial train/test split.
- Bootstrap summarized the mean and SD of the simple model's error across different training resamples.
- Adjusted R² compared two models using the same target and accounted for predictor count.

**What I learned:** High scores are not enough by themselves. We need to understand where features come from, use proper held-out data, and check whether performance is stable. In particular, CVSS attributes and severity are closely related by design, so severity classification here should not be mistaken for a general prediction of cybersecurity risk.